In [1]:
!pip install -f http://h2o-release.s3.amazonaws.com/h2o/latest_stable_Py.html h2o

Looking in links: http://h2o-release.s3.amazonaws.com/h2o/latest_stable_Py.html
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 MB 5.1 MB/s eta 0:00:00


In [3]:
# Load common libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import numpy as np

# Framework libraries
import h2o
from h2o.automl import H2OAutoML
h2o.init()

Checking whether there is an H2O instance running at http://localhost:54321. connected.
Please download and install the latest version from: https://h2o-release.s3.amazonaws.com/h2o/latest_stable.html


H2O_cluster_uptime:,20 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.7
H2O_cluster_version_age:,6 months and 8 days
H2O_cluster_name:,H2O_from_python_unknownUser_47jlz6
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.168 Gb
H2O_cluster_total_cores:,2
H2O_cluster_allowed_cores:,2
H2O_cluster_status:,"locked, healthy"


In [12]:
# Reading dataset
data = pd.read_csv("train.csv")
data["CabinNoCabin"] = np.where(data["Cabin"].isnull(), "No Cabin", "Cabin")
data["survived"] = np.where(data["Survived"] == 1, "yes", "no")
# data["Age"] = data["Age"].fillna(data["Age"].mean())
data.drop(["PassengerId", "Name", "Ticket", "Cabin", "Survived"], axis = 1, inplace = True)
data.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,CabinNoCabin,survived
0,3,male,22.0,1,0,7.2500,S,No Cabin,no
1,1,female,38.0,1,0,71.2833,C,Cabin,yes
2,3,female,26.0,0,0,7.9250,S,No Cabin,yes
3,1,female,35.0,1,0,53.1000,S,Cabin,yes
4,3,male,35.0,0,0,8.0500,S,No Cabin,no


In [13]:
# Split data
train_data, test_data = train_test_split(data, test_size = 0.2)

In [14]:
# Create datasets in the right form for H2O
train_data = h2o.H2OFrame(train_data)
test_data = h2o.H2OFrame(test_data)

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


In [15]:
# Role definition
X = train_data.columns
y = 'survived'
X.remove(y)

In [16]:
# Set up framework
automl = H2OAutoML(max_runtime_secs=300, seed = 1)

In [17]:
# Fit model
automl.train(x = X, y=y, training_frame=train_data)

AutoML progress: |███████████████████████████████████████████████████████████████| (done) 100%


key,value
Stacking strategy,cross_validation
Number of base models (used / total),51/51
# GBM base models (used / total),22/22
# XGBoost base models (used / total),17/17
# DeepLearning base models (used / total),9/9
# DRF base models (used / total),2/2
# GLM base models (used / total),1/1
Metalearner algorithm,GBM
Metalearner fold assignment scheme,Random
Metalearner nfolds,5


In [18]:
lb = automl.leaderboard
lb.head(rows = lb.nrows)

model_id,auc,logloss,aucpr,mean_per_class_error,rmse,mse
StackedEnsemble_AllModels_5_AutoML_2_20251006_24550,0.872271,0.425117,0.825333,0.185756,0.365143,0.13333
StackedEnsemble_BestOfFamily_4_AutoML_2_20251006_24550,0.868701,0.416975,0.842461,0.174566,0.360385,0.129877
GBM_lr_annealing_selection_AutoML_2_20251006_24550_select_model,0.868655,0.420345,0.843528,0.177299,0.361122,0.130409
GBM_grid_1_AutoML_2_20251006_24550_model_15,0.866359,0.433446,0.838029,0.18004,0.368538,0.13582
StackedEnsemble_BestOfFamily_2_AutoML_2_20251006_24550,0.865368,0.422406,0.838062,0.176174,0.36171,0.130834
StackedEnsemble_AllModels_2_AutoML_2_20251006_24550,0.865022,0.41813,0.843632,0.176157,0.359062,0.128926
GBM_grid_1_AutoML_2_20251006_24550_model_17,0.864872,0.423089,0.841374,0.187365,0.363398,0.132058
GBM_grid_1_AutoML_2_20251006_24550_model_2,0.864759,0.41873,0.841578,0.174116,0.360152,0.12971
StackedEnsemble_BestOfFamily_3_AutoML_2_20251006_24550,0.864422,0.422989,0.837932,0.177774,0.36157,0.130733
GBM_grid_1_AutoML_2_20251006_24550_model_5,0.864393,0.426749,0.840586,0.180974,0.363315,0.131998


In [19]:
# Show best model
best_model = automl.leader
best_model

key,value
Stacking strategy,cross_validation
Number of base models (used / total),51/51
# GBM base models (used / total),22/22
# XGBoost base models (used / total),17/17
# DeepLearning base models (used / total),9/9
# DRF base models (used / total),2/2
# GLM base models (used / total),1/1
Metalearner algorithm,GBM
Metalearner fold assignment scheme,Random
Metalearner nfolds,5


In [20]:
# Evaluate performance on both traind and test set
performance_train = best_model.model_performance(test_data = train_data)
performance_test = best_model.model_performance(test_data = test_data)

In [21]:
# Report statistics
print("Train Accuracy score: ", performance_train.accuracy())
print("Train AUC score: ", performance_train.auc())

print("Test Accuracy score: ", performance_test.accuracy())
print("Test AUC score: ", performance_test.auc())

Train Accuracy score:  [[0.4383746211142458, 0.8946629213483146]]
Train AUC score:  0.9210995567109956
Test Accuracy score:  [[0.30239445910527846, 0.8435754189944135]]
Test AUC score:  0.8739401165871753


In [22]:
# Save the best model
model_path = '/content/drive/MyDrive/Colab Notebooks/Kaggle/h2omodel_Titanic01'
h2o.save_model(model=best_model, path = model_path, force = True)

'/content/drive/MyDrive/Colab Notebooks/Kaggle/h2omodel_Titanic01/StackedEnsemble_AllModels_5_AutoML_2_20251006_24550'

In [ ]:
print(automl.feature_importance(data))

Computing feature importance via permutation shuffling for 8 features using 891 rows with 5 shuffle sets...
	3.39s	= Expected runtime (0.68s per shuffle set)
	0.85s	= Actual runtime (Completed 5 of 5 shuffle sets)


              importance    stddev   p_value  n  p99_high   p99_low
Sex             0.204040  0.020130  0.000011  5  0.245489  0.162592
Pclass          0.102581  0.008346  0.000005  5  0.119766  0.085397
Age             0.101908  0.007751  0.000004  5  0.117868  0.085948
Fare            0.078563  0.009323  0.000023  5  0.097759  0.059368
SibSp           0.020875  0.003773  0.000123  5  0.028644  0.013107
CabinNoCabin    0.010101  0.003722  0.001863  5  0.017765  0.002437
Parch           0.005387  0.002905  0.007150  5  0.011369 -0.000594
Embarked        0.005387  0.001463  0.000594  5  0.008400  0.002374


# PREDICTION

In [23]:
# Reading new data
new_data = pd.read_csv("test.csv")
new_data["CabinNoCabin"] = np.where(new_data["Cabin"].isnull(), "No Cabin", "Cabin")
new_data.drop(["PassengerId", "Name", "Ticket", "Cabin"], axis = 1, inplace = True)

new_data = h2o.H2OFrame(new_data)

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


In [25]:
# Read saved model
loaded_model = h2o.load_model('/content/drive/MyDrive/Colab Notebooks/Kaggle/h2omodel_Titanic01/StackedEnsemble_AllModels_5_AutoML_2_20251006_24550')
loaded_model

key,value
Stacking strategy,cross_validation
Number of base models (used / total),51/51
# GBM base models (used / total),22/22
# XGBoost base models (used / total),17/17
# DeepLearning base models (used / total),9/9
# DRF base models (used / total),2/2
# GLM base models (used / total),1/1
Metalearner algorithm,GBM
Metalearner fold assignment scheme,Random
Metalearner nfolds,5


In [26]:
# Predict new dataset
prediction = loaded_model.predict(new_data)
prediction

stackedensemble prediction progress: |███████████████████████████████████████████| (done) 100%


predict,no,yes
no,0.807319,0.192681
no,0.671003,0.328997
no,0.725192,0.274808
no,0.884,0.116
yes,0.245502,0.754498
no,0.886555,0.113445
yes,0.423773,0.576227
no,0.868888,0.131112
yes,0.288778,0.711222
no,0.927545,0.0724547


In [27]:
h2o.export_file(prediction, "survived01_h2omodel.csv")

Export File progress: |██████████████████████████████████████████████████████████| (done) 100%


# FINDING THE CUT-OFF POINT

In [54]:
"""
y_def = test_data.as_data_frame(use_pandas=True)["survived"]
y_true = np.where(y_def == "yes", 1, 0)
y_true = pd.DataFrame(y_true)
y_true["survived"] = y_true[0]
y_true.drop([0], axis = 1, inplace = True)
y_true
"""

'\ny_def = test_data.as_data_frame(use_pandas=True)["survived"]\ny_true = np.where(y_def == "yes", 1, 0)\ny_true = pd.DataFrame(y_true)\ny_true["survived"] = y_true[0]\ny_true.drop([0], axis = 1, inplace = True)\ny_true\n'

In [55]:
# df = pd.read_csv("test.csv")
# df

In [56]:
h2o.shutdown()

H2O session _sid_9a7f closed.


/tmp/ipython-input-2900054706.py:1: H2ODeprecationWarning: Deprecated, use ``h2o.cluster().shutdown()``.
  h2o.shutdown()
